In [3]:
## arxiv--research paper tool
from langchain_community.tools import WikipediaQueryRun,ArxivQueryRun
from langchain_community.utilities import WikipediaAPIWrapper,ArxivAPIWrapper


In [4]:
api_wrapper_wikipedia=WikipediaAPIWrapper(top_k_results=3,doc_content_chars_max=250)
wiki = WikipediaQueryRun(api_wrapper=api_wrapper_wikipedia)

api_wrapper_arxiv = ArxivAPIWrapper(top_k_results=1,doc_content_chars_max=250)
arxiv=ArxivQueryRun(api_wrapper=api_wrapper_arxiv)

In [5]:
print(arxiv.name)

arxiv


In [6]:
arxiv

ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.ArxivError'>, <class 'arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.HTTPError'>), top_k_results=1, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=250))

In [7]:
tools=[wiki,arxiv]

In [8]:
## custom tools rag tools
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
 

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [11]:
loader = WebBaseLoader("https://arxiv.org/abs/1706.03762")
docs = loader.load()
documents = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200).split_documents(docs)
vector_db=FAISS.from_documents(documents,HuggingFaceEmbeddings())
retriever= vector_db.as_retriever()
retriever

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2713.48it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001EB28478CD0>, search_kwargs={})

In [16]:
from langchain_core.tools import create_retriever_tool
retriever_tool = create_retriever_tool(
    retriever, 
    "langsmith-search", 
    "Search any information about Transformer"
)

retriever_tool.name

'langsmith-search'

In [18]:

tool=[wiki,arxiv,retriever_tool]
tool

[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'd:\\GEN AI Courses\\GEN AI PROJECTS\\myenv\\Lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=3, lang='en', load_all_available_meta=False, doc_content_chars_max=250)),
 ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.ArxivError'>, <class 'arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.HTTPError'>), top_k_results=1, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=250)),
 StructuredTool(name='langsmith-search', description='Search any information about Transformer', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=<function create_retriever_tool.<locals>.func at 0x000001EB3B3296C0>, coroutine=<function create_retriever_tool.<locals>.afunc at 0x000001EB3B3299E0>)]

In [35]:
## Run all this tools with agents and llm models
from langchain_groq import ChatGroq
from dotenv import load_dotenv
load_dotenv
import os
groq_api_key = os.getenv("GROQ_API_KEY")
hf_api_key=os.getenv("HF_TOKEN")
llm = ChatGroq(groq_api_key=groq_api_key,model_name="openai/gpt-oss-120b")

In [28]:
## prompt template
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
prompt =  ChatPromptTemplate(
    [
        ("system", "You are a helpful AI bot. Your name is KD."),
        ("human", "Hello, how are you doing?"),
        ("ai", "I'm doing well, thanks!"),
        ("human", "{user_input}"),
        MessagesPlaceholder(variable_name="agent_scratchpad"), 
    ]
)
prompt.messages

[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a helpful AI bot. Your name is KD.'), additional_kwargs={}),
 HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='Hello, how are you doing?'), additional_kwargs={}),
 AIMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template="I'm doing well, thanks!"), additional_kwargs={}),
 HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['user_input'], input_types={}, partial_variables={}, template='{user_input}'), additional_kwargs={}),
 MessagesPlaceholder(variable_name='agent_scratchpad')]

In [37]:
#agents 
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent
agent = create_tool_calling_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools,verbose=True,handle_parsing_errors=True )
agent_executor
agent_executor.invoke({
    "user_input": "what is paper 1706.03762 about ??"
})



> Entering new AgentExecutor chain...

Invoking: `arxiv` with `{'query': '1706.03762'}`


Published: 2023-08-02
Title: Attention Is All You Need
Authors: Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Lukasz Kaiser, Illia Polosukhin
Summary: The dominant sequence transduction models are based on c**Paper:** *Attention Is All You Need* (arXiv: 1706.03762)  
**Authors:** Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, Illia Polosukhin  
**Year:** 2017 (presented at NIPS 2017)

---

## TL;DR
The paper introduces the **Transformer**, a neural network architecture for sequence‑to‑sequence tasks that **replaces recurrence and convolution with pure self‑attention**. This design enables highly parallelizable training, reduces the path length between any two positions in a sequence, and achieves state‑of‑the‑art results on machine translation while using fewer computational resources than prior R

{'user_input': 'what is paper 1706.03762 about ??',
 'output': '**Paper:** *Attention Is All You Need* (arXiv:\u202f1706.03762)  \n**Authors:** Ashish\u202fVaswani, Noam\u202fShazeer, Niki\u202fParmar, Jakob\u202fUszkoreit, Llion\u202fJones, Aidan\u202fN.\u202fGomez, Łukasz\u202fKaiser, Illia\u202fPolosukhin  \n**Year:** 2017 (presented at NIPS 2017)\n\n---\n\n## TL;DR\nThe paper introduces the **Transformer**, a neural network architecture for sequence‑to‑sequence tasks that **replaces recurrence and convolution with pure self‑attention**. This design enables highly parallelizable training, reduces the path length between any two positions in a sequence, and achieves state‑of‑the‑art results on machine translation while using fewer computational resources than prior RNN/CNN models.\n\n---\n\n## 1. Motivation\n\n* **Recurrent Neural Networks (RNNs)** and **Convolutional Neural Networks (CNNs)** dominate sequence modeling but have drawbacks:  \n  * RNNs process tokens sequentially → lim